# Experiment 1 — fresh execution and equality investigation

**Reproducibility rerun after test inspection.** This notebook invokes the original, hash-verified `experiments/train.py` runner. It trains league → Arsenal → fine-tuning, preserving every setting and frozen input. Python tracing records each epoch without changing the training function, random state, gradients or optimizer.

Each `fit` resets Python, NumPy and PyTorch to seed 42 and enables deterministic algorithms. Consequently, changing the order of independent fits does not change their initialization, minibatch permutations or dropout draws. The league and Arsenal models are different models; equality is checked between corresponding models across runs, not between Arsenal and league.

Run from the repository checkout with its pinned requirements. CPU PyTorch 2.5.1+cpu was used. No archived predictions or checkpoints initialize the independent models; only fine-tuning loads the league checkpoint produced by this execution.

In [1]:
from pathlib import Path
import sys,json
PROJECT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'experiments/train.py').exists())
sys.path.insert(0,str(PROJECT))
import pandas as pd
from IPython.display import display
from experiments.rerun_experiment_1 import run
RUN=run()
print('Fresh output directory:',RUN)

league_mlp epoch=001 train=1.087098465 validation=1.068020344 best_epoch=1 patience=0/20 steps=32


league_mlp epoch=002 train=1.076027052 validation=1.047609329 best_epoch=2 patience=0/20 steps=64


league_mlp epoch=003 train=1.064099416 validation=1.023564696 best_epoch=3 patience=0/20 steps=96


league_mlp epoch=004 train=1.051737241 validation=0.998163402 best_epoch=4 patience=0/20 steps=128


league_mlp epoch=005 train=1.040044806 validation=0.972663283 best_epoch=5 patience=0/20 steps=160


league_mlp epoch=006 train=1.028716886 validation=0.949228346 best_epoch=6 patience=0/20 steps=192


... (212 outputs left out)

league_to_arsenal_mlp epoch=019 train=0.897291842 validation=0.799001515 best_epoch=1 patience=18/20 steps=76


league_to_arsenal_mlp epoch=020 train=0.901485599 validation=0.799062073 best_epoch=1 patience=19/20 steps=80


league_to_arsenal_mlp epoch=021 train=0.903576806 validation=0.799120426 best_epoch=1 patience=20/20 steps=84


{
  "label": "Experiment 1 reproducibility rerun after test inspection",
  "started_utc": "2026-10-26T00:31:58.840868+00:00",
  "finished_utc": "2026-10-26T00:32:03.677326+00:00",
  "original_training_source_sha256": "3af682854e7fde017f3780ed49749431f2675e19ee7fa990ccd68f85c41d7898",
  "rerun_observer_source_sha256": "6b9dcd5a3c800a91e2654852cf2b2d685287e3c6f469cc70a97191830f83d7cf",
  "original_training_source_unchanged": true,
  "execution_order": [
    "league_mlp",
    "arsenal_mlp",
    "league_to_arsenal_mlp"
  ],
  "observed_epochs": 221,
  "optimizer_updates": 3684,
  "max_probability_difference": 0.0,
  "all_weights_exactly_equal": true,
  "all_epoch_losses_exactly_equal": true
}
Fresh output directory: /workspace/Premier-League-Research/results/run_20261026T003158


## Compare fresh results with the original and experiment 2

In [2]:
display(pd.Series(json.loads((RUN/'rerun_verification.json').read_text())))
display(pd.read_csv(RUN/'prediction_comparison.csv'))
display(pd.read_csv(RUN/'weights_and_epochs_comparison.csv'))
metrics=pd.read_csv(RUN/'metrics.csv')
display(metrics.query("split == 'test' and population == 'Arsenal'"))

label                                 Experiment 1 reproducibility rerun after test ...
started_utc                                            2026-10-26T00:31:58.840868+00:00
finished_utc                                           2026-10-26T00:32:03.677326+00:00
original_training_source_sha256       3af682854e7fde017f3780ed49749431f2675e19ee7fa9...
rerun_observer_source_sha256          6b9dcd5a3c800a91e2654852cf2b2d685287e3c6f469cc...
original_training_source_unchanged                                                 True
execution_order                        [league_mlp, arsenal_mlp, league_to_arsenal_mlp]
observed_epochs                                                                     221
optimizer_updates                                                                  3684
max_probability_difference                                                          0.0
all_weights_exactly_equal                                                          True
all_epoch_losses_exactly_equal  

,reference,model,split,max_probability_difference
0,experiment_1_original,league_mlp,validation,0.0
1,experiment_1_original,league_mlp,test,0.0
2,experiment_1_original,arsenal_mlp,validation,0.0
3,experiment_1_original,arsenal_mlp,test,0.0
4,experiment_1_original,league_to_arsenal_mlp,validation,0.0
5,experiment_1_original,league_to_arsenal_mlp,test,0.0
6,experiment_1_original,league_logistic,validation,0.0
7,experiment_1_original,league_logistic,test,0.0
8,experiment_1_original,arsenal_logistic,validation,0.0
9,experiment_1_original,arsenal_logistic,test,0.0


,reference,model,weights_exactly_equal,epoch_losses_exactly_equal
0,experiment_1_original,league_mlp,True,True
1,experiment_2_latest,league_mlp,True,True
2,experiment_1_original,arsenal_mlp,True,True
3,experiment_2_latest,arsenal_mlp,True,True
4,experiment_1_original,league_to_arsenal_mlp,True,True
5,experiment_2_latest,league_to_arsenal_mlp,True,True


,model,split,population,log_loss,brier_multiclass,accuracy,n
18,league_mlp,test,Arsenal,0.899941,0.529087,0.613333,75
21,arsenal_mlp,test,Arsenal,0.959723,0.569044,0.600000,75
24,league_to_arsenal_mlp,test,Arsenal,0.900410,0.529357,0.613333,75
27,league_logistic,test,Arsenal,0.908957,0.532644,0.613333,75
30,arsenal_logistic,test,Arsenal,0.909300,0.534445,0.640000,75
33,league_prior,test,Arsenal,1.094381,0.664183,0.400000,75


## Every epoch
Training loss is the online minibatch average with dropout; validation is evaluated after the epoch with dropout disabled. Test is scored after fitting and validation-only model selection. The tables are intentionally complete.

In [3]:
epochs=pd.read_csv(RUN/'observed_epochs.csv')
for model,table in epochs.groupby('model',sort=False):
    print(model)
    with pd.option_context('display.max_rows',None,'display.max_columns',None):
        display(table.drop(columns=['model','observed_utc']).round(9))

league_mlp


,epoch,training_log_loss,validation_arsenal_log_loss,best_epoch_so_far,patience_used,optimizer_steps
0,1,1.087098,1.068020,1,0,32
1,2,1.076027,1.047609,2,0,64
2,3,1.064099,1.023565,3,0,96
3,4,1.051737,0.998163,4,0,128
4,5,1.040045,0.972663,5,0,160
5,6,1.028717,0.949228,6,0,192
6,7,1.019569,0.928597,7,0,224
7,8,1.011865,0.911380,8,0,256
8,9,1.006175,0.897544,9,0,288
9,10,1.003229,0.886274,10,0,320


arsenal_mlp


,epoch,training_log_loss,validation_arsenal_log_loss,best_epoch_so_far,patience_used,optimizer_steps
100,1,1.089138,1.081805,1,0,4
101,2,1.086166,1.079540,2,0,8
102,3,1.084127,1.077312,3,0,12
103,4,1.082826,1.075056,4,0,16
104,5,1.080569,1.072784,5,0,20
105,6,1.079655,1.070475,6,0,24
106,7,1.076868,1.068149,7,0,28
107,8,1.074225,1.065802,8,0,32
108,9,1.072349,1.063411,9,0,36
109,10,1.073181,1.060987,10,0,40


league_to_arsenal_mlp


,epoch,training_log_loss,validation_arsenal_log_loss,best_epoch_so_far,patience_used,optimizer_steps
200,1,0.906066,0.796373,1,0,4
201,2,0.901878,0.796407,1,1,8
202,3,0.909410,0.796657,1,2,12
203,4,0.904080,0.796975,1,3,16
204,5,0.907378,0.797208,1,4,20
205,6,0.905636,0.797432,1,5,24
206,7,0.904114,0.797634,1,6,28
207,8,0.901585,0.797771,1,7,32
208,9,0.896478,0.797894,1,8,36
209,10,0.902493,0.798029,1,9,40


## Interpretation
Exact equality across corresponding models is an expected consequence of this deterministic setup. It does not add independent evidence of model performance. Changing only execution order is a workflow change, not a different statistical treatment. A multi-seed study would answer a different question and would require explicitly changing the fixed-seed protocol.